## 1. Install Libraries

In [1]:
!pip install -q pyarrow
!pip install -q tqdm

## 2. Imports

In [2]:
import os
import gc
import re
import json
import time
import warnings

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

## 3. Project Paths

In [3]:
PROJECT_PATH = "/content/drive/MyDrive/FinSight_AI"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data"
)

PROCESSED_PATH = os.path.join(
    DATA_PATH,
    "processed"
)

CLASSIFICATION_PATH = os.path.join(
    DATA_PATH,
    "classification"
)

CLASSIFICATION_BATCH_PATH = os.path.join(
    CLASSIFICATION_PATH,
    "classification_batches"
)

REPORT_PATH = os.path.join(
    PROJECT_PATH,
    "reports"
)

os.makedirs(
    CLASSIFICATION_BATCH_PATH,
    exist_ok=True
)

## 4. Load Dataset

In [4]:
financial_news = pd.read_parquet(

    os.path.join(

        PROCESSED_PATH,

        "financial_news_clean.parquet"

    ),

    columns=[

        "news_id",

        "headline"

    ]

)

print(financial_news.shape)

financial_news.head()

(3215296, 2)


,news_id,headline
0,1,Stocks That Hit 52-Week Highs On Friday
1,2,Stocks That Hit 52-Week Highs On Wednesday
2,3,71 Biggest Movers From Friday
3,4,46 Stocks Moving In Friday's Mid-Day Session
4,5,B of A Securities Maintains Neutral on Agilent...


## 5. Financial Event Taxonomy

In [5]:
EVENT_CATEGORIES = [

    "Earnings",
    "Revenue Guidance",
    "Analyst Rating",
    "Dividend",
    "Stock Split",
    "Share Buyback",
    "Merger & Acquisition",
    "Partnership",
    "IPO",
    "Funding",
    "Executive Change",
    "Layoffs",
    "Product Launch",
    "Regulatory Approval",
    "Litigation",
    "Cybersecurity",
    "Fraud",
    "Bankruptcy",
    "Credit Rating",
    "Monetary Policy",
    "Economic Data",
    "Commodity",
    "Cryptocurrency",
    "ESG",

    "Market Movement",
    "Technical Analysis",
    "Options Activity",
    "Trading Halt",
    "Insider Trading",

    "Other"

]

print("="*70)
print("Financial Event Categories")
print("="*70)

for i, event in enumerate(EVENT_CATEGORIES,1):

    print(f"{i:02d}. {event}")

print()

print(f"Total Categories : {len(EVENT_CATEGORIES)}")

Financial Event Categories
01. Earnings
02. Revenue Guidance
03. Analyst Rating
04. Dividend
05. Stock Split
06. Share Buyback
07. Merger & Acquisition
08. Partnership
09. IPO
10. Funding
11. Executive Change
12. Layoffs
13. Product Launch
14. Regulatory Approval
15. Litigation
16. Cybersecurity
17. Fraud
18. Bankruptcy
19. Credit Rating
20. Monetary Policy
21. Economic Data
22. Commodity
23. Cryptocurrency
24. ESG
25. Market Movement
26. Technical Analysis
27. Options Activity
28. Trading Halt
29. Insider Trading
30. Other

Total Categories : 30


## 6. Market Signal Categories

In [6]:
MARKET_SIGNALS = [

    "Bullish",

    "Bearish",

    "Neutral",

    "Mixed"

]

print(MARKET_SIGNALS)

['Bullish', 'Bearish', 'Neutral', 'Mixed']


In [7]:
EVENT_KEYWORDS = {

    "Earnings":[

        "earnings",
        "earnings preview",
        "earnings call",
        "earnings conference",
        "earnings transcript",
        "eps",
        "revenue",
        "results",
        "quarterly",
        "guidance",
        "forecast",
        "beat",
        "beats",
        "miss",
        "misses",
        "estimate",
        "estimates"

    ],

    "Revenue Guidance": [
        "raises guidance", "lowers guidance", "guidance",
        "forecast", "outlook", "expects", "projects"
    ],

    "Analyst Rating": [

        "upgrades",
        "upgrade",
        "downgrade",
        "downgrades",
        "maintains",
        "initiates",
        "target price",
        "price target",
        "raises target",
        "lowers target",
        "buy rating",
        "sell rating",
        "neutral",
        "overweight",
        "underweight",
        "outperform",
        "underperform",
        "equal-weight",
        "equal weight"

    ],

    "Dividend": [
        "dividend", "cash dividend",
        "special dividend", "declares dividend",
        "quarterly dividend"
    ],

    "Stock Split": [
        "stock split", "reverse split",
        "share split"
    ],

    "Share Buyback": [
        "buyback", "share repurchase",
        "repurchase", "repurchase program",
        "buyback program"
    ],

    "Merger & Acquisition": [
        "acquires", "acquire", "acquisition",
        "merger", "merges", "buyout",
        "takeover"
    ],

    "Partnership": [
        "partnership", "partners",
        "collaboration", "joint venture",
        "agreement", "strategic alliance"
    ],

    "IPO": [
        "ipo", "initial public offering",
        "goes public", "listed",
        "listing"
    ],

    "Funding": [
        "funding", "investment",
        "raises capital", "series a",
        "series b", "venture capital",
        "financing"
    ],

    "Executive Change": [
        "ceo", "cfo", "chairman",
        "appoints", "appointment",
        "resigns", "retires",
        "steps down"
    ],

    "Layoffs": [
        "layoffs", "job cuts",
        "cuts jobs", "workforce reduction",
        "reduces workforce"
    ],

    "Product Launch": [
        "launches", "launch",
        "introduces", "introducing",
        "unveils", "announces"
    ],

    "Regulatory Approval": [
        "approval", "approved",
        "clearance", "authorized",
        "fda", "sec"
    ],

    "Litigation": [
        "lawsuit", "court",
        "legal", "settlement",
        "sues", "appeal"
    ],

    "Cybersecurity": [
        "hack", "breach",
        "cyber", "cyberattack",
        "ransomware", "malware",
        "data leak"
    ],

    "Fraud": [
        "fraud", "investigation",
        "probe", "misconduct",
        "scandal"
    ],

    "Bankruptcy": [
        "bankruptcy", "chapter 11",
        "insolvency", "liquidation"
    ],

    "Credit Rating": [
        "credit rating", "moody",
        "fitch", "s&p",
        "rating"
    ],

    "Monetary Policy": [
        "federal reserve", "fed",
        "interest rate", "rate hike",
        "rate cut", "fomc"
    ],

    "Economic Data": [
        "inflation", "cpi",
        "ppi", "gdp",
        "unemployment"
    ],

    "Commodity": [
        "oil", "crude",
        "gold", "silver",
        "natural gas", "opec"
    ],

    "Cryptocurrency": [
        "bitcoin", "ethereum",
        "crypto", "blockchain"
    ],

    "ESG": [
        "esg", "sustainability",
        "carbon", "net zero"
    ],

    "Market Movement":[

        "52-week high",
        "52-week low",
        "stocks moving",
        "shares trading",
        "trading higher",
        "trading lower",
        "biggest movers",
        "top gainers",
        "top losers",
        "market update",
        "mid-day session",
        "mid morning market update",
        "mid-afternoon market update",
        "pre-market",
        "after hours",
        "after-hours",
        "market recap",
        "opening bell",
        "closing bell"

    ],

    "Technical Analysis": [
        "support", "resistance",
        "breakout", "bullish",
        "bearish", "moving average",
        "rsi", "macd"
    ],

    "Options Activity":[

        "option alert",
        "options alert",
        "unusual options",
        "options activity",
        "call options",
        "put options",
        "calls",
        "puts"

    ],

    "Trading Halt": [
        "trading halted",
        "halted",
        "volatility halt"
    ],

    "Insider Trading": [
        "insider", "director bought",
        "director sold", "officer bought",
        "officer sold", "form 4"
    ]

}

print(f"Total Event Classes : {len(EVENT_KEYWORDS)}")

Total Event Classes : 29


## 7. Event Keyword Dictionary

In [8]:
import re
from collections import defaultdict

# Higher weight = stronger evidence
KEYWORD_WEIGHTS = {

    "earnings":3,
    "eps":3,
    "revenue":2,
    "profit":2,
    "loss":2,
    "guidance":3,
    "forecast":2,
    "dividend":3,
    "stock split":4,
    "share repurchase":4,
    "buyback":4,
    "merger":4,
    "acquisition":4,
    "acquires":4,
    "partnership":3,
    "collaboration":3,
    "ipo":5,
    "bankruptcy":5,
    "chapter 11":5,
    "fraud":5,
    "hack":5,
    "breach":5,
    "lawsuit":4,
    "fda":4,
    "approval":4,
    "interest rate":5,
    "fed":4,
    "inflation":4,
    "bitcoin":4,
    "oil":3,
    "gold":3
}


def classify_headline(headline):

    text = headline.lower()

    scores = defaultdict(float)

    for category, keywords in EVENT_KEYWORDS.items():

        for keyword in keywords:

            if re.search(r"\b" + re.escape(keyword) + r"\b", text):

                scores[category] += KEYWORD_WEIGHTS.get(keyword,1)

    if len(scores)==0:

        return {

            "category":"Other",

            "confidence":0.0,

            "matched_keywords":[]

        }

    best_category = max(

        scores,

        key=scores.get

    )

    total = sum(scores.values())

    confidence = scores[best_category]/total

    matched=[]

    for keyword in EVENT_KEYWORDS[best_category]:

        if re.search(r"\b"+re.escape(keyword)+r"\b",text):

            matched.append(keyword)

    return {

        "category":best_category,

        "confidence":round(confidence,3),

        "matched_keywords":matched

    }

## 8. Bullish Signal Dictionary

In [9]:
BULLISH_KEYWORDS = {

    "beat",
    "beats",
    "surges",
    "rises",
    "gains",
    "growth",
    "strong",
    "record",
    "raises",
    "approved",
    "approval",
    "buyback",
    "dividend",
    "upgrade",
    "outperform",
    "overweight",
    "expands",
    "launches",
    "wins",
    "profit",
    "higher",
    "positive"

}

print(len(BULLISH_KEYWORDS))

22


## 9. Bearish Signal Dictionary

In [10]:
BEARISH_KEYWORDS = {

    "miss",
    "misses",
    "falls",
    "drops",
    "declines",
    "cut",
    "cuts",
    "lower",
    "downgrade",
    "underperform",
    "underweight",
    "lawsuit",
    "fraud",
    "bankruptcy",
    "breach",
    "hack",
    "layoffs",
    "loss",
    "warning",
    "investigation",
    "negative"

}

print(len(BEARISH_KEYWORDS))

21


## 10. Keyword Weights

In [11]:
KEYWORD_WEIGHTS = {

    # Strong financial events
    "ipo": 5,
    "bankruptcy": 5,
    "chapter 11": 5,
    "merger": 5,
    "acquisition": 5,
    "stock split": 5,
    "share repurchase": 5,
    "buyback": 5,

    # Strong regulation
    "approval": 4,
    "fda": 4,
    "lawsuit": 4,
    "fraud": 4,

    # Earnings
    "earnings": 4,
    "eps": 4,
    "guidance": 4,

    # Ratings
    "upgrade": 3,
    "downgrade": 3,
    "maintains": 2,

    # Market movement
    "52-week high": 3,
    "52-week low": 3,
    "biggest movers": 3,
    "stocks moving": 3
}

KEYWORD_WEIGHTS.update({

    "earnings preview":4,
    "earnings transcript":5,
    "earnings conference":4,

    "option alert":5,
    "options alert":5,
    "unusual options":5,

    "52-week high":5,
    "52-week low":5,

    "market update":4,
    "market recap":4,

    "top gainers":5,
    "top losers":5,
    "biggest movers":5,

    "raises target":4,
    "lowers target":4,
    "price target":4,

    "maintains":3,
    "upgrade":4,
    "downgrade":4

})

print(f"Weighted Keywords : {len(KEYWORD_WEIGHTS)}")

Weighted Keywords : 35


### EVENT_PRIORITY

In [12]:
EVENT_PRIORITY = {

    "Bankruptcy": 100,

    "Fraud": 95,

    "Cybersecurity": 94,

    "Litigation": 92,

    "Regulatory Approval": 90,

    "Merger & Acquisition": 88,

    "IPO": 86,

    "Funding": 84,

    "Executive Change": 82,

    "Layoffs": 80,

    "Share Buyback": 78,

    "Dividend": 76,

    "Stock Split": 74,

    "Earnings": 72,

    "Revenue Guidance": 70,

    "Analyst Rating": 68,

    "Partnership": 66,

    "Product Launch": 64,

    "Credit Rating": 62,

    "Monetary Policy": 60,

    "Economic Data": 58,

    "Commodity": 56,

    "Cryptocurrency": 54,

    "Market Movement": 52,

    "Technical Analysis": 50,

    "Options Activity": 48,

    "Trading Halt": 46,

    "Insider Trading": 44,

    "ESG": 42,

    "Other": 0

}

In [13]:
SPECIAL_PATTERNS = {

    "Layoffs":[

        r"cuts?\s+\d+\s+jobs",
        r"cuts?\s+\d+\s+employees",
        r"layoffs?",
        r"job\s+cuts?",
        r"reduces?\s+workforce",
        r"workforce\s+reduction"

    ],

    "Market Movement":[

        r"52[- ]week\s+highs?",
        r"52[- ]week\s+lows?",
        r"stocks?\s+moving",
        r"shares?\s+trading",
        r"biggest\s+movers",
        r"top\s+gainers",
        r"top\s+losers",
        r"mid[- ]day\s+session",
        r"mid[- ]morning\s+market\s+update",
        r"mid[- ]afternoon\s+market\s+update",
        r"after[- ]hours?",
        r"pre[- ]market"

    ],

    "Options Activity":[

        r"option\s+alert",
        r"options\s+activity",
        r"unusual\s+options",

    ],

    "Analyst Rating":[

        r"maintains?\s+\w+\s+on",
        r"upgrades?\s+\w+\s+to",
        r"downgrades?\s+\w+\s+to"

    ]

}

## 11. Multi-Score Rule Engine

In [14]:
import re
from collections import defaultdict

def classify_headline(headline):

    text = str(headline).lower()

    event_scores = defaultdict(float)
    matched_keywords = defaultdict(list)

    bullish_score = 0
    bearish_score = 0

    # 1. Regex Pattern Matching (Higher Priority)
    for event, patterns in SPECIAL_PATTERNS.items():

        for pattern in patterns:

            if re.search(pattern, text):

                event_scores[event] += 5

                matched_keywords[event].append(pattern)

    # 2. Keyword Matching
    for event, keywords in EVENT_KEYWORDS.items():

        for keyword in keywords:

            pattern = r"\b" + re.escape(keyword) + r"\b"

            if re.search(pattern, text):

                weight = KEYWORD_WEIGHTS.get(keyword, 1)

                event_scores[event] += weight

                matched_keywords[event].append(keyword)

    # 3. Bullish Signal
    for word in BULLISH_KEYWORDS:

        if re.search(r"\b" + re.escape(word) + r"\b", text):

            bullish_score += 1

    # 4. Bearish Signal
    for word in BEARISH_KEYWORDS:

        if re.search(r"\b" + re.escape(word) + r"\b", text):

            bearish_score += 1

    # 5. No Rule Matched
    if len(event_scores) == 0:

        return {

            "event": "Other",

            "market_signal": "Neutral",

            "confidence": 0.0,

            "matched_keywords": "",

            "rule_score": 0,

            "method": "Rule"

        }

    # 6. Select Best Event

    # Select Best Event (Score + Priority)
    best_event = sorted(

        event_scores.items(),

        key=lambda x: (

            x[1],                          # Rule Score

            EVENT_PRIORITY.get(x[0], 0)    # Priority

        ),

        reverse=True

    )[0][0]

    best_score = event_scores[best_event]

    total_score = sum(event_scores.values())

    confidence = best_score / total_score

    # 7. Market Signal
    if best_event in [

        "Monetary Policy",

        "Economic Data",

        "Market Movement"

    ]:

        signal = "Neutral"

    else:

        if bullish_score > bearish_score:

            signal = "Bullish"

        elif bearish_score > bullish_score:

            signal = "Bearish"

        elif bullish_score == bearish_score == 0:

            signal = "Neutral"

        else:

            signal = "Mixed"

    # 8. Return
    return {

        "event": best_event,

        "market_signal": signal,

        "confidence": round(confidence, 3),

        "matched_keywords": ", ".join(

            sorted(

                set(

                    matched_keywords[best_event]

                )

            )

        ),

        "rule_score": best_score,

        "method": "Rule"

    }

## 12. Test the Rule Engine

In [15]:
examples = [

    "Apple reports record quarterly earnings and raises guidance",

    "Microsoft acquires Activision Blizzard",

    "Federal Reserve raises interest rates",

    "Tesla announces stock split",

    "JPMorgan declares quarterly dividend",

    "Coinbase suffers data breach",

    "Pfizer receives FDA approval",

    "Intel cuts 5000 jobs",

    "Stocks That Hit 52-Week Highs On Friday",

    "B of A Securities Maintains Neutral on Agilent"

]

for headline in examples:

    result = classify_headline(headline)

    print("="*80)

    print("Headline :")

    print(headline)

    print()

    for k, v in result.items():

        print(f"{k:18}: {v}")

Headline :
Apple reports record quarterly earnings and raises guidance

event             : Earnings
market_signal     : Bullish
confidence        : 0.643
matched_keywords  : earnings, guidance, quarterly
rule_score        : 9.0
method            : Rule
Headline :
Microsoft acquires Activision Blizzard

event             : Merger & Acquisition
market_signal     : Neutral
confidence        : 1.0
matched_keywords  : acquires
rule_score        : 1.0
method            : Rule
Headline :
Federal Reserve raises interest rates

event             : Monetary Policy
market_signal     : Neutral
confidence        : 1.0
matched_keywords  : federal reserve
rule_score        : 1.0
method            : Rule
Headline :
Tesla announces stock split

event             : Stock Split
market_signal     : Neutral
confidence        : 0.833
matched_keywords  : stock split
rule_score        : 5.0
method            : Rule
Headline :
JPMorgan declares quarterly dividend

event             : Dividend
market_signal   

## 13. Batch Configuration

In [16]:
BATCH_SIZE = 5000

TOTAL_ROWS = len(financial_news)

TOTAL_BATCHES = (

    TOTAL_ROWS + BATCH_SIZE - 1

) // BATCH_SIZE

print("="*70)

print("Financial News Classification")

print("="*70)

print(f"Total Rows      : {TOTAL_ROWS:,}")

print(f"Batch Size      : {BATCH_SIZE:,}")

print(f"Total Batches   : {TOTAL_BATCHES}")

Financial News Classification
Total Rows      : 3,215,296
Batch Size      : 5,000
Total Batches   : 644


## 14. Output Paths

In [17]:
RULE_OUTPUT_PATH = os.path.join(

    CLASSIFICATION_PATH,

    "rule_classification"

)

RULE_BATCH_PATH = os.path.join(

    RULE_OUTPUT_PATH,

    "classification_batches"

)

os.makedirs(

    RULE_BATCH_PATH,

    exist_ok=True

)

print(RULE_BATCH_PATH)

/content/drive/MyDrive/FinSight_AI/data/classification/rule_classification/classification_batches


## 15. Checkpoint Functions

In [18]:
CHECKPOINT_FILE = os.path.join(

    RULE_OUTPUT_PATH,

    "classification_checkpoint.json"

)

def save_checkpoint(

    batch_number,

    processed_rows

):

    checkpoint = {

        "last_batch": batch_number,

        "processed_rows": processed_rows

    }

    with open(

        CHECKPOINT_FILE,

        "w"

    ) as f:

        json.dump(

            checkpoint,

            f,

            indent=4

        )


def load_checkpoint():

    if os.path.exists(

        CHECKPOINT_FILE

    ):

        with open(

            CHECKPOINT_FILE,

            "r"

        ) as f:

            return json.load(f)

    return {

        "last_batch":0,

        "processed_rows":0

    }

## 16. Process One Batch

In [19]:
def process_batch(batch_df):

    results = []

    for row in batch_df.itertuples(index=False):

        prediction = classify_headline(

            row.headline

        )

        results.append({

            "news_id": row.news_id,

            "headline": row.headline,

            "event": prediction["event"],

            "market_signal": prediction["market_signal"],

            "confidence": prediction["confidence"],

            "matched_keywords": prediction["matched_keywords"],

            "rule_score": prediction["rule_score"],

            "method": prediction["method"]

        })

    return pd.DataFrame(results)

## 17. Test One Batch

In [20]:
test_batch = financial_news.iloc[:20]

test_output = process_batch(

    test_batch

)

display(test_output)

,news_id,headline,event,market_signal,confidence,matched_keywords,rule_score,method
0,1,Stocks That Hit 52-Week Highs On Friday,Market Movement,Neutral,1.000,52[- ]week\s+highs?,5.0,Rule
1,2,Stocks That Hit 52-Week Highs On Wednesday,Market Movement,Neutral,1.000,52[- ]week\s+highs?,5.0,Rule
2,3,71 Biggest Movers From Friday,Market Movement,Neutral,1.000,"biggest movers, biggest\s+movers",10.0,Rule
3,4,46 Stocks Moving In Friday's Mid-Day Session,Market Movement,Neutral,1.000,"mid-day session, mid[- ]day\s+session, stocks ...",14.0,Rule
4,5,B of A Securities Maintains Neutral on Agilent...,Analyst Rating,Bullish,1.000,"maintains, maintains?\s+\w+\s+on, neutral, pri...",13.0,Rule
5,6,"CFRA Maintains Hold on Agilent Technologies, L...",Analyst Rating,Neutral,1.000,"maintains, maintains?\s+\w+\s+on, price target",12.0,Rule
6,7,"UBS Maintains Neutral on Agilent Technologies,...",Analyst Rating,Bullish,1.000,"maintains, maintains?\s+\w+\s+on, neutral, pri...",13.0,Rule
7,8,Agilent Technologies shares are trading higher...,Earnings,Bullish,0.833,"eps, results",5.0,Rule
8,9,Wells Fargo Maintains Overweight on Agilent Te...,Analyst Rating,Bullish,1.000,"maintains, maintains?\s+\w+\s+on, overweight, ...",13.0,Rule
9,10,10 Biggest Price Target Changes For Friday,Analyst Rating,Neutral,1.000,price target,4.0,Rule


## 18. Load Checkpoint

In [21]:
checkpoint = load_checkpoint()

START_BATCH = checkpoint["last_batch"]

processed_rows = checkpoint["processed_rows"]

print("="*70)

print("Checkpoint Loaded")

print("="*70)

print(f"Start Batch : {START_BATCH}")

print(f"Processed Rows : {processed_rows:,}")

Checkpoint Loaded
Start Batch : 644
Processed Rows : 3,215,296


## 19. Process Entire Dataset

In [22]:
overall_start = time.time()

print("="*70)

print("Processing Entire Dataset")

print("="*70)

for batch_number in tqdm(

    range(

        START_BATCH,

        TOTAL_BATCHES

    ),

    desc="Financial Classification"

):

    output_file = os.path.join(

        RULE_BATCH_PATH,

        f"classification_batch_{batch_number+1:04d}.parquet"

    )

    if os.path.exists(

        output_file

    ):

        continue

    start = batch_number * BATCH_SIZE

    end = min(

        start + BATCH_SIZE,

        TOTAL_ROWS

    )

    batch_df = financial_news.iloc[

        start:end

    ]

    batch_start = time.time()

    batch_result = process_batch(

        batch_df

    )

    batch_result.to_parquet(

        output_file,

        index=False

    )

    processed_rows += len(

        batch_result

    )

    save_checkpoint(

        batch_number + 1,

        processed_rows

    )

    elapsed = time.time() - overall_start

    avg_batch = elapsed / (batch_number + 1)

    eta = (

        TOTAL_BATCHES

        - batch_number

        - 1

    ) * avg_batch / 60

    if (

        (batch_number + 1) % 25 == 0

        or

        batch_number == TOTAL_BATCHES - 1

    ):

        print()

        print("="*70)

        print(f"Batch : {batch_number+1}/{TOTAL_BATCHES}")

        print(f"Rows Processed : {processed_rows:,}")

        print(f"ETA : {eta:.2f} minutes")

        print(f"Batch Time : {time.time()-batch_start:.2f} sec")

        print(f"Average Confidence : {batch_result['confidence'].mean():.3f}")

    del batch_df
    del batch_result

    gc.collect()

print()

print("="*70)

print("RULE CLASSIFICATION COMPLETED")

print("="*70)

print(f"Total Rows : {processed_rows:,}")

Processing Entire Dataset


Financial Classification: 0it [00:00, ?it/s]


RULE CLASSIFICATION COMPLETED
Total Rows : 3,215,296


## 20. Merge All Batch Files

In [23]:
classification_files = sorted(

    os.listdir(

        RULE_BATCH_PATH

    )

)

rule_df = pd.concat(

    (

        pd.read_parquet(

            os.path.join(

                RULE_BATCH_PATH,

                file

            )

        )

        for file in tqdm(

            classification_files,

            desc="Merging Batches"

        )

    ),

    ignore_index=True

)

OUTPUT_FILE = os.path.join(

    RULE_OUTPUT_PATH,

    "news_classification_rule.parquet"

)

rule_df.to_parquet(

    OUTPUT_FILE,

    index=False

)

print()

print("="*70)

print("MASTER FILE CREATED")

print("="*70)

print(rule_df.shape)

rule_df.head()

Merging Batches:   0%|          | 0/644 [00:00<?, ?it/s]


MASTER FILE CREATED
(3215296, 8)


,news_id,headline,event,market_signal,confidence,matched_keywords,rule_score,method
0,1,Stocks That Hit 52-Week Highs On Friday,Market Movement,Neutral,1.0,52[- ]week\s+highs?,5.0,Rule
1,2,Stocks That Hit 52-Week Highs On Wednesday,Market Movement,Neutral,1.0,52[- ]week\s+highs?,5.0,Rule
2,3,71 Biggest Movers From Friday,Market Movement,Neutral,1.0,"biggest movers, biggest\s+movers",10.0,Rule
3,4,46 Stocks Moving In Friday's Mid-Day Session,Market Movement,Neutral,1.0,"mid-day session, mid[- ]day\s+session, stocks ...",14.0,Rule
4,5,B of A Securities Maintains Neutral on Agilent...,Analyst Rating,Bullish,1.0,"maintains, maintains?\s+\w+\s+on, neutral, pri...",13.0,Rule


## 21. Evaluation

In [24]:
category_summary = (

    rule_df

    .groupby("event")

    .agg(

        headlines=("news_id","count"),

        avg_confidence=("confidence","mean")

    )

    .sort_values(

        "headlines",

        ascending=False

    )

    .reset_index()

)

display(category_summary)

,event,headlines,avg_confidence
0,Other,1584926,0.000000
1,Earnings,565184,0.945028
2,Analyst Rating,240475,0.924911
3,Market Movement,190472,0.971070
4,Dividend,109350,0.959246
5,Commodity,86252,0.944960
6,Merger & Acquisition,55997,0.928586
7,Product Launch,44348,0.977842
8,Executive Change,43360,0.926570
9,Partnership,34528,0.909540


## 22. Summary

In [25]:
summary = pd.DataFrame({

    "Metric":[

        "Rows",

        "Events",

        "Batch Files",

        "Output"

    ],

    "Value":[

        len(rule_df),

        rule_df["event"].nunique(),

        len(classification_files),

        os.path.basename(

            OUTPUT_FILE

        )

    ]

})

display(summary)

,Metric,Value
0,Rows,3215296
1,Events,30
2,Batch Files,644
3,Output,news_classification_rule.parquet


## 23. Create Unresolved Dataset

In [29]:
# Only keep headlines that the rule engine could not classify

unresolved_df = rule_df[

    rule_df["event"] == "Other"

].copy()

unresolved_df = unresolved_df.sort_values(

    "news_id"

).reset_index(

    drop=True

)

OUTPUT_UNRESOLVED = os.path.join(

    RULE_OUTPUT_PATH,

    "unresolved_news.parquet"

)

unresolved_df.to_parquet(

    OUTPUT_UNRESOLVED,

    index=False

)

print("=" * 70)
print("UNRESOLVED NEWS CREATED")
print("=" * 70)

print(f"Rows : {len(unresolved_df):,}")

print(f"Percentage : {100 * len(unresolved_df) / len(rule_df):.2f}%")

print(f"Output File : {os.path.basename(OUTPUT_UNRESOLVED)}")

print()

unresolved_df.head()

UNRESOLVED NEWS CREATED
Rows : 1,584,926
Percentage : 49.29%
Output File : unresolved_news.parquet



,news_id,headline,event,market_signal,confidence,matched_keywords,rule_score,method
0,18,"Q1 13F Roundup: How Buffett, Einhorn, Ackman A...",Other,Neutral,0.0,,0.0,Rule
1,19,Pershing Square 13F Shows Fund Raises Stake In...,Other,Neutral,0.0,,0.0,Rule
2,20,How Bill Ackman Successfully Navigated Coronav...,Other,Neutral,0.0,,0.0,Rule
3,25,Agilent Reports Has Become Top-Level Sponsor O...,Other,Neutral,0.0,,0.0,Rule
4,54,"Q4 13F Roundup: How Buffett, Einhorn, Ackman A...",Other,Neutral,0.0,,0.0,Rule


## 24. Statistics

In [30]:
summary = pd.DataFrame({

    "Metric": [

        "Total Headlines",

        "Resolved by Rule Engine",

        "Unresolved (Transformer Input)",

        "Resolved Percentage",

        "Unresolved Percentage"

    ],

    "Value": [

        len(rule_df),

        len(rule_df) - len(unresolved_df),

        len(unresolved_df),

        round(

            100 * (len(rule_df) - len(unresolved_df)) / len(rule_df),

            2

        ),

        round(

            100 * len(unresolved_df) / len(rule_df),

            2

        )

    ]

})

display(summary)

,Metric,Value
0,Total Headlines,3215296.00
1,Resolved by Rule Engine,1630370.00
2,Unresolved (Transformer Input),1584926.00
3,Resolved Percentage,50.71
4,Unresolved Percentage,49.29


## 25. Confidence Distribution

In [31]:
confidence_summary = (

    rule_df

    .groupby("event")

    .agg(

        headlines=("news_id","count"),

        avg_confidence=("confidence","mean"),

        min_confidence=("confidence","min"),

        max_confidence=("confidence","max")

    )

    .sort_values(

        "headlines",

        ascending=False

    )

    .reset_index()

)

display(confidence_summary)

,event,headlines,avg_confidence,min_confidence,max_confidence
0,Other,1584926,0.000000,0.000,0.0
1,Earnings,565184,0.945028,0.200,1.0
2,Analyst Rating,240475,0.924911,0.200,1.0
3,Market Movement,190472,0.971070,0.286,1.0
4,Dividend,109350,0.959246,0.250,1.0
5,Commodity,86252,0.944960,0.333,1.0
6,Merger & Acquisition,55997,0.928586,0.200,1.0
7,Product Launch,44348,0.977842,0.333,1.0
8,Executive Change,43360,0.926570,0.200,1.0
9,Partnership,34528,0.909540,0.250,1.0
